# Single-Agent vs Multi-Agent Trade-offs

The most common architecture decision in agent engineering is not *whether* to use AI — it is **how many agents** to use:

- **Single agent + many tools** — one LLM loop with a broad tool belt.
- **Multi-agent system (MAS)** — several specialists coordinated by a supervisor or pipeline.

This notebook helps you choose deliberately. You will study a **trade-off matrix** across cost, latency, reliability, and debuggability; walk a **decision flowchart**; and run the **same real task** through both architectures so you can see concrete differences — not just theory.

**Scenario:** A team needs to produce a **customer onboarding guide** for a SaaS product — pulling facts from internal docs, drafting prose, and passing a compliance review before publish.

Everything is self-contained below. No prior notebooks or frameworks are required.

In [ ]:
"""
Shared environment: onboarding guide task — used by both single-agent and multi-agent implementations.
"""

import json
import os
import re
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

INTERNAL_DOCS: list[dict[str, str]] = [
    {
        "id": "prod-01",
        "title": "Product overview",
        "text": "Acme Analytics provides dashboards, scheduled reports, and team workspaces. Free tier includes 3 seats.",
    },
    {
        "id": "sec-02",
        "title": "Security and compliance",
        "text": "SOC 2 Type II certified. Do not include customer PII in onboarding emails. MFA required for admin roles.",
    },
    {
        "id": "setup-03",
        "title": "Account setup steps",
        "text": "1. Verify email. 2. Invite teammates. 3. Connect data source. 4. Run sample dashboard template.",
    },
    {
        "id": "billing-04",
        "title": "Billing FAQ",
        "text": "Pro plan is $49/seat/month billed annually. 14-day trial converts automatically unless cancelled.",
    },
]

COMPLIANCE_RULES = {
    "forbidden_phrases": ["guaranteed ROI", "100% secure", "no risk"],
    "required_sections": ["getting started", "security", "billing"],
    "must_mention_mfa": True,
}

print(f"Internal docs: {len(INTERNAL_DOCS)}")
print(f"Compliance rules loaded: {len(COMPLIANCE_RULES['required_sections'])} required sections")

---

## 1. The Central Question

```
                    ┌─────────────────────┐
                    │  One agent enough?  │
                    └──────────┬──────────┘
                          yes  │  no
                    ┌───────────┴───────────┐
                    ▼                       ▼
         single agent + tools          multi-agent design
         (simpler ops)                 (roles + coordinator)
```

Both approaches use LLMs and tools. The difference is **how many role-specific prompts and coordination layers** you add on top.

There is no universal winner — only a fit for your constraints: budget, latency SLA, governance requirements, and team ability to operate the system.

---

## 2. Trade-off Matrix

| Dimension | Single agent + tools | Multi-agent system |
|-----------|----------------------|--------------------|
| **Cost** | Fewer LLM calls, one system prompt | More calls + supervisor routing overhead |
| **Latency** | Fewer hops, simpler path | Extra routing turns; parallel workers can help |
| **Reliability** | Single failure domain | Partial progress possible; critic can catch errors |
| **Debuggability** | Linear trace — easy to read | Multi-thread transcripts — richer but harder |
| **Prompt clarity** | Can become bloated with many roles | Focused per-role prompts |
| **Parallelism** | Mostly sequential tool loop | Independent specialists can run in parallel |
| **Governance** | Hard to embed a true veto | Dedicated critic / security agent |
| **Operational complexity** | Lower — one agent to tune | Higher — topology, handoffs, termination |
| **Onboarding (human)** | Easier for small teams | Requires understanding roles and routing |

Use this matrix in design reviews. Score what matters most for **your** product before defaulting to multi-agent because it sounds advanced.

---

## 3. Cost — Tokens and LLM Calls

**Single agent:** one system prompt + alternating assistant/tool messages until done.

```
Single:  [system][user][assistant][tool][assistant][tool][assistant]
```

**Multi-agent:** each specialist may have its own system prompt; the supervisor adds routing calls and handoff messages.

```
Multi:   [supervisor system][route][worker1 system][work][supervisor][worker2 system][work]...
```

Mitigations for multi-agent cost:

- Use a **smaller/cheaper model** for routing only.
- **Summarize** shared context instead of passing full transcripts.
- **Cache** retrieval results in shared state so workers do not re-search.

In [ ]:
@dataclass
class CostEstimate:
    """Rough token/call model for architecture comparison (not billing-accurate)."""

    system_prompt_tokens: int = 500
    per_turn_tokens: int = 800
    supervisor_overhead_tokens: int = 400

    def single_agent(self, tool_turns: int) -> dict[str, int]:
        calls = 1 + tool_turns  # initial + one LLM call per tool cycle
        tokens = self.system_prompt_tokens + calls * self.per_turn_tokens
        return {"llm_calls": calls, "est_tokens": tokens}

    def multi_agent(
        self, workers: int, supervisor_turns: int, tools_per_worker: int = 1
    ) -> dict[str, int]:
        worker_calls = workers * tools_per_worker
        supervisor_calls = supervisor_turns
        calls = worker_calls + supervisor_calls
        tokens = (
            workers * self.system_prompt_tokens
            + supervisor_calls * self.supervisor_overhead_tokens
            + calls * self.per_turn_tokens
        )
        return {"llm_calls": calls, "est_tokens": tokens}


estimator = CostEstimate()
single = estimator.single_agent(tool_turns=4)
multi = estimator.multi_agent(workers=3, supervisor_turns=4, tools_per_worker=1)

print(f"{'Architecture':<20} {'LLM calls':>10} {'Est. tokens':>12}")
print("-" * 44)
print(f"{'Single agent':<20} {single['llm_calls']:>10} {single['est_tokens']:>12}")
print(f"{'Multi-agent':<20} {multi['llm_calls']:>10} {multi['est_tokens']:>12}")
print(f"\nMulti-agent overhead: {multi['est_tokens'] - single['est_tokens']:+d} tokens (estimate)")

---

## 4. Latency — Wall Clock Time

| Factor | Single agent | Multi-agent |
|--------|--------------|-------------|
| LLM round-trips | Fewer sequential calls | More calls (routing between workers) |
| Parallel work | Tools run one after another in loop | Researchers can run in parallel |
| Cold start | One model session | Multiple role contexts to warm |
| User-facing chat | Usually faster p95 | Often slower unless parallelized |

For interactive assistants with tight SLAs (e.g. under 3 seconds), **single agent** is often the right default. For batch jobs (overnight report generation), multi-agent latency is more acceptable.

In [ ]:
def simulate_latency(
    llm_calls: int,
    ms_per_call: int = 400,
    parallel_workers: int = 1,
) -> dict[str, float]:
    """Simulate wall-clock latency from LLM call count and parallelism."""
    if parallel_workers > 1:
        worker_batches = max(1, llm_calls // parallel_workers)
        wall_ms = worker_batches * ms_per_call
    else:
        wall_ms = llm_calls * ms_per_call
    return {"llm_calls": llm_calls, "wall_ms": wall_ms, "wall_sec": round(wall_ms / 1000, 2)}


single_lat = simulate_latency(single["llm_calls"], parallel_workers=1)
multi_lat = simulate_latency(multi["llm_calls"], parallel_workers=1)
multi_parallel = simulate_latency(4, parallel_workers=2)  # 2 researchers in parallel

print(f"Single agent:     {single_lat['wall_sec']}s ({single_lat['llm_calls']} calls)")
print(f"Multi-agent:      {multi_lat['wall_sec']}s ({multi_lat['llm_calls']} calls)")
print(f"Multi (parallel): {multi_parallel['wall_sec']}s (4 calls, 2 parallel workers)")

---

## 5. Reliability and Debuggability

### Reliability — failure modes

| Failure | Single agent | Multi-agent |
|---------|--------------|-------------|
| Bad tool call | Retry in same loop | Worker retries; critic may catch before publish |
| Role confusion | Common when prompt is huge | Reduced with focused role prompts |
| Partial progress | Often all-or-nothing | Researcher draft may survive writer failure |
| Infinite loop | `max_steps` on one loop | Per-agent caps + global supervisor cap |

### Debuggability

- **Single agent:** one ordered trace — ideal for beginners and incident response.
- **Multi-agent:** supervisor decisions + worker spans — harder to read but shows *who* made each mistake.

Production tip: log `agent_name` on every span, whether single or multi.

---

## 6. Shared Tools — Used by Both Architectures

Before comparing architectures, define the **capabilities** both need: search docs, draft guide, compliance check.

In [ ]:
def search_docs(query: str, top_k: int = 3) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in INTERNAL_DOCS:
        text = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for term in terms if term in text) if terms else 1
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [{"id": d["id"], "title": d["title"], "text": d["text"]} for _, d in scored[:top_k]]


def draft_onboarding_guide(facts: dict[str, Any]) -> str:
    changes = facts.get("setup_steps", [])
    security = facts.get("security_note", "MFA required for admins.")
    billing = facts.get("billing_note", "Pro plan billed annually.")
    return (
        f"# Getting started with Acme Analytics\n\n"
        f"Welcome! Follow these steps: {' → '.join(changes) if changes else 'see setup guide'}.\n\n"
        f"## Security\n{security}\n\n"
        f"## Billing\n{billing}\n"
    )


def compliance_check(draft: str) -> dict[str, Any]:
    issues = []
    lower = draft.lower()
    for phrase in COMPLIANCE_RULES["forbidden_phrases"]:
        if phrase in lower:
            issues.append(f"forbidden phrase: '{phrase}'")
    for section in COMPLIANCE_RULES["required_sections"]:
        if section not in lower:
            issues.append(f"missing section: '{section}'")
    if COMPLIANCE_RULES["must_mention_mfa"] and "mfa" not in lower:
        issues.append("must mention MFA")
    return {"approved": len(issues) == 0, "issues": issues}


# Quick tool sanity check
hits = search_docs("security MFA")
print(f"Search 'security MFA' → [{hits[0]['id']}] {hits[0]['title']}")

---

## 7. Single-Agent Implementation

One agent performs **search → draft → compliance check** in a single loop. This wins when the workflow is linear and governance is light.

In [ ]:
@dataclass
class RunMetrics:
    architecture: str
    steps: int = 0
    llm_calls: int = 0
    tool_calls: int = 0
    approved: bool = False
    trace: list[str] = field(default_factory=list)
    elapsed_ms: float = 0.0

    def log(self, msg: str) -> None:
        self.trace.append(msg)
        self.steps += 1


@dataclass
class SingleAgentOnboarding:
    """One agent, three tools, one loop — search, draft, check."""

    max_steps: int = 6

    def run(self, goal: str) -> tuple[str, RunMetrics]:
        start = time.perf_counter()
        m = RunMetrics(architecture="single_agent")
        m.llm_calls = 1  # one system prompt session
        m.log(f"GOAL: {goal}")

        # Tool 1: search (simulates LLM deciding to call search_docs)
        m.llm_calls += 1
        hits = search_docs("setup security billing")
        m.tool_calls += 1
        m.log(f"tool:search_docs → {len(hits)} documents")

        facts = {
            "setup_steps": ["Verify email", "Invite teammates", "Connect data source"],
            "security_note": next((h["text"] for h in hits if "sec" in h["id"]), "MFA required."),
            "billing_note": next((h["text"] for h in hits if "billing" in h["id"]), "See pricing page."),
        }

        # Tool 2: draft
        m.llm_calls += 1
        draft = draft_onboarding_guide(facts)
        m.tool_calls += 1
        m.log(f"tool:draft_guide → {len(draft.split())} words")

        # Tool 3: compliance
        m.llm_calls += 1
        review = compliance_check(draft)
        m.tool_calls += 1
        m.log(f"tool:compliance_check → approved={review['approved']}")

        m.approved = review["approved"]
        m.elapsed_ms = (time.perf_counter() - start) * 1000
        return draft, m


single_agent = SingleAgentOnboarding()
single_draft, single_metrics = single_agent.run("Create customer onboarding guide")

print(f"Approved: {single_metrics.approved}")
print(f"Steps: {single_metrics.steps} | LLM calls: {single_metrics.llm_calls} | Tools: {single_metrics.tool_calls}")
print(f"Trace: {single_metrics.trace}")

---

## 8. Multi-Agent Implementation

Specialists split the work: **researcher** gathers facts, **writer** drafts, **compliance** agent has **veto power**. A **supervisor** routes between them.

This wins when:

- Security/compliance needs **conflicting** instructions from the writer ("be marketing-friendly" vs "never overclaim").
- You want an explicit **reject → revise** loop.
- Different roles may later use **different models** (cheap researcher, capable writer).

In [ ]:
@dataclass
class MultiAgentState:
    artifacts: dict[str, Any] = field(default_factory=dict)
    messages: list[str] = field(default_factory=list)

    def log(self, agent: str, msg: str) -> None:
        self.messages.append(f"[{agent}] {msg}")


class ResearcherWorker:
    name = "researcher"
    def run(self, state: MultiAgentState) -> None:
        hits = search_docs("setup security billing product")
        state.artifacts["facts"] = {
            "docs": hits,
            "setup_steps": ["Verify email", "Invite teammates", "Connect data source"],
            "security_note": next((h["text"] for h in hits if "sec" in h["id"]), ""),
            "billing_note": next((h["text"] for h in hits if "billing" in h["id"]), ""),
        }
        state.log(self.name, f"gathered {len(hits)} docs")


class WriterWorker:
    name = "writer"
    def run(self, state: MultiAgentState) -> None:
        facts = state.artifacts.get("facts", {})
        draft = draft_onboarding_guide(facts)
        state.artifacts["draft"] = draft
        state.log(self.name, f"draft ready ({len(draft.split())} words)")


class ComplianceWorker:
    name = "compliance"
    def run(self, state: MultiAgentState) -> None:
        draft = state.artifacts.get("draft", "")
        review = compliance_check(draft)
        state.artifacts["review"] = review
        state.log(self.name, f"verdict: approved={review['approved']}, issues={review.get('issues', [])}")


WORKERS = {
    "researcher": ResearcherWorker(),
    "writer": WriterWorker(),
    "compliance": ComplianceWorker(),
}


def supervisor_next(state: MultiAgentState) -> str:
    if "facts" not in state.artifacts:
        return "researcher"
    if "draft" not in state.artifacts:
        return "writer"
    if "review" not in state.artifacts:
        return "compliance"
    if state.artifacts["review"].get("approved"):
        return "FINISH"
    # One revision loop
    if state.artifacts.get("revision_count", 0) < 1:
        state.artifacts["revision_count"] = 1
        state.artifacts.pop("draft", None)
        state.artifacts.pop("review", None)
        return "writer"
    return "FINISH"


@dataclass
class MultiAgentOnboarding:
    max_turns: int = 10

    def run(self, goal: str) -> tuple[str, RunMetrics]:
        start = time.perf_counter()
        m = RunMetrics(architecture="multi_agent")
        state = MultiAgentState()
        state.log("supervisor", f"GOAL: {goal}")

        for _ in range(self.max_turns):
            m.llm_calls += 1  # supervisor routing call
            nxt = supervisor_next(state)
            state.log("supervisor", f"route → {nxt}")
            m.log(f"supervisor routes → {nxt}")

            if nxt == "FINISH":
                break

            worker = WORKERS[nxt]
            m.llm_calls += 1  # worker LLM call
            worker.run(state)
            m.tool_calls += 1
            m.log(f"worker:{worker.name} completed")

        draft = state.artifacts.get("draft", "")
        review = state.artifacts.get("review", {})
        m.approved = review.get("approved", False)
        m.elapsed_ms = (time.perf_counter() - start) * 1000
        return draft, m


multi_agent = MultiAgentOnboarding()
multi_draft, multi_metrics = multi_agent.run("Create customer onboarding guide")

print(f"Approved: {multi_metrics.approved}")
print(f"Steps: {multi_metrics.steps} | LLM calls: {multi_metrics.llm_calls} | Tools: {multi_metrics.tool_calls}")
for line in multi_metrics.trace:
    print(f"  {line}")

---

## 9. Side-by-Side Comparison — Same Task, Both Architectures

Running the identical onboarding goal through both implementations makes trade-offs concrete.

In [ ]:
def compare_architectures(single_m: RunMetrics, multi_m: RunMetrics) -> None:
    print(f"{'Metric':<22} {'Single':>12} {'Multi':>12} {'Delta':>12}")
    print("-" * 60)
    rows = [
        ("LLM calls", single_m.llm_calls, multi_m.llm_calls),
        ("Tool/worker calls", single_m.tool_calls, multi_m.tool_calls),
        ("Trace steps", single_m.steps, multi_m.steps),
        ("Approved", int(single_m.approved), int(multi_m.approved)),
        ("Elapsed (ms)", int(single_m.elapsed_ms), int(multi_m.elapsed_ms)),
    ]
    for label, s, mu in rows:
        delta = mu - s if isinstance(s, int) and isinstance(mu, int) else "—"
        print(f"{label:<22} {s:>12} {mu:>12} {delta:>12}")


compare_architectures(single_metrics, multi_metrics)

---

## 10. When Single Agent + Many Tools Wins

| Signal | Onboarding example |
|--------|-------------------|
| **Linear workflow** | Search docs → draft → done (no veto loop) |
| **Shared context** | Same product facts for all steps |
| **Low role conflict** | One "helpful onboarding assistant" persona |
| **Tight latency** | In-app tooltip or live chat |
| **MVP / small team** | Ship one ReAct agent before building a crew |

**Simple doc Q&A** — *"What are the setup steps?"* — is almost always single-agent + search tool.

In [ ]:
def simple_doc_qa(question: str) -> str:
    """Single-agent path: one search, one answer — no crew needed."""
    hits = search_docs(question, top_k=1)
    if not hits:
        return "No documentation found."
    h = hits[0]
    return f"[{h['id']}] {h['title']}: {h['text']}"


print(simple_doc_qa("How do I set up my account?"))

---

## 11. When Multi-Agent Wins

| Signal | Onboarding example |
|--------|-------------------|
| **Conflicting goals** | Marketing tone vs compliance strictness |
| **Parallel research** | Product docs + security + billing researched separately |
| **Strong governance** | Compliance agent **blocks** publish until approved |
| **Org-shaped roles** | Research / write / legal map to real teams |
| **Long-horizon deliverables** | Full onboarding pack with multiple sections |

Below, a **bad draft** with a forbidden phrase shows why a dedicated compliance agent matters.

In [ ]:
risky_draft = (
    "# Getting started\n\n"
    "Acme Analytics is 100% secure with guaranteed ROI!\n\n"
    "## Billing\nPro plan details here."
)

single_review = compliance_check(risky_draft)
print("Compliance review of risky draft:")
print(f"  Approved: {single_review['approved']}")
print(f"  Issues: {single_review['issues']}")
print("\nIn multi-agent design, compliance agent VETO prevents publish until writer revises.")

---

## 12. Decision Flowchart (Implemented)

```
 START
   │
   ▼
 Is the task decomposable into independent expert roles?
   │no                          │yes
   ▼                             ▼
 Can one system prompt cover      Do roles need conflicting
 all tools + policies?            instructions or veto power?
   │yes       │no                  │yes       │no
   ▼          ▼                     ▼          ▼
 SINGLE     Try prompt split      MULTI      Try sequential
 AGENT      or memory first       AGENT      single-agent chain
   │                             │
   └─────────────┬───────────────┘
                 ▼
         Set max_steps, evals, cost caps
                 │
                 ▼
              SHIP + MEASURE
```

In [ ]:
def architecture_decision(
    independent_roles: bool,
    prompt_fits: bool,
    needs_veto: bool,
    latency_sensitive: bool = False,
) -> str:
    if latency_sensitive and not needs_veto:
        return "single_agent (latency-sensitive)"
    if not independent_roles:
        return "single_agent" if prompt_fits else "single_agent + memory / split prompts"
    if needs_veto:
        return "multi_agent with critic + supervisor"
    if not prompt_fits:
        return "multi_agent or hybrid supervisor"
    return "multi_agent OR sequential single-agent chain — prototype both"


SCENARIOS = [
    ("Account setup FAQ", False, True, False, True),
    ("Full onboarding pack", True, False, True, False),
    ("List pricing tiers", False, True, False, True),
    ("Compliance-heavy launch email", True, False, True, False),
    ("Translate UI string", False, True, False, True),
]

print(f"{'Scenario':<28} {'Recommendation'}")
print("-" * 60)
for name, ir, pf, nv, lat in SCENARIOS:
    rec = architecture_decision(ir, pf, nv, lat)
    print(f"{name:<28} {rec}")

---

## 13. Hybrid Pattern — Supervisor + Single-Tool Workers

Production systems often land in the **middle**: a supervisor routes to **subgraphs** that are internally single-agent.

```
 supervisor
    ├── docs subgraph (1 agent, search only)
    └── compliance subgraph (1 agent, check only)
```

You get role separation and auditability without multiplying LLM calls unnecessarily inside each subgraph.

In [ ]:
@dataclass
class HybridOrchestrator:
    """Supervisor routes to single-purpose worker subgraphs."""

    def docs_subgraph(self, query: str) -> list[dict[str, str]]:
        return search_docs(query)

    def writer_subgraph(self, facts: dict[str, Any]) -> str:
        return draft_onboarding_guide(facts)

    def compliance_subgraph(self, draft: str) -> dict[str, Any]:
        return compliance_check(draft)

    def run(self, goal: str) -> dict[str, Any]:
        trace = ["supervisor: start"]
        hits = self.docs_subgraph("setup security billing")
        trace.append(f"docs_subgraph: {len(hits)} hits")
        facts = {
            "setup_steps": ["Verify email", "Invite teammates"],
            "security_note": hits[1]["text"] if len(hits) > 1 else "",
            "billing_note": hits[-1]["text"],
        }
        draft = self.writer_subgraph(facts)
        trace.append(f"writer_subgraph: {len(draft.split())} words")
        review = self.compliance_subgraph(draft)
        trace.append(f"compliance_subgraph: approved={review['approved']}")
        return {"draft": draft, "review": review, "trace": trace}


hybrid = HybridOrchestrator()
hybrid_out = hybrid.run("Onboarding guide")
for step in hybrid_out["trace"]:
    print(step)

---

## 14. Tool Proliferation vs Agent Proliferation

| Approach | Risk | Mitigation |
|----------|------|------------|
| **Many tools, one agent** | Tool selection errors — model picks wrong API | Group tools; improve descriptions; reduce count |
| **Many agents, few tools each** | Routing errors — supervisor sends to wrong worker | Clear role boundaries; eval routing accuracy |

**Rules of thumb:**

- More than **~8 similar tools** → consider splitting into agents by domain.
- More than **~3 distinct domains** (docs, billing, compliance) → consider multi-agent or hybrid supervisor.
- If **routing accuracy** falls below your eval threshold → add supervisor or improve tool schemas before adding more agents.

In [ ]:
def proliferation_advice(tool_count: int, domain_count: int) -> str:
    if domain_count >= 3 and tool_count >= 8:
        return "multi-agent or hybrid supervisor recommended"
    if tool_count >= 8:
        return "split tools across agents OR reduce tool surface"
    if domain_count >= 3:
        return "consider domain-specific agents"
    return "single agent + tools is likely sufficient"


for tools, domains in [(4, 1), (10, 2), (12, 4)]:
    print(f"tools={tools}, domains={domains} → {proliferation_advice(tools, domains)}")

---

## 15. Migration Path — Start Single, Split Later

A practical rollout that avoids premature complexity:

1. **MVP** — single ReAct agent with search + draft + check tools.
2. **Measure** — tool selection accuracy, compliance miss rate, latency p95.
3. **Extract** — if compliance misses exceed threshold, promote check to dedicated agent.
4. **Add supervisor** when routing between domains becomes the bottleneck.

The simulation below models **when to split** based on error rate.

In [ ]:
@dataclass
class MigrationSimulator:
    tool_error_rate: float = 0.05
    compliance_miss_rate: float = 0.12
    split_threshold: float = 0.10

    def recommend_split(self) -> dict[str, Any]:
        actions = []
        if self.compliance_miss_rate > self.split_threshold:
            actions.append("extract compliance_critic agent")
        if self.tool_error_rate > self.split_threshold:
            actions.append("split tools by domain under supervisor")
        if not actions:
            actions.append("stay single-agent; continue measuring")
        return {
            "tool_error_rate": self.tool_error_rate,
            "compliance_miss_rate": self.compliance_miss_rate,
            "actions": actions,
        }


sim = MigrationSimulator(tool_error_rate=0.05, compliance_miss_rate=0.12)
result = sim.recommend_split()
print(json.dumps(result, indent=2))

---

## 16. Feature Comparison Table

| Feature | Recommended architecture | Why |
|---------|-------------------------|-----|
| Doc Q&A | Single + search | Linear, read-only, low risk |
| Create draft from chat | Single + draft tool | Short chain |
| Full onboarding pack | Multi or hybrid | Parallel research + compliance veto |
| Compliance-heavy launch | Multi + human approval | Governance + legal risk |
| Autocomplete field | Neither — UI function | Deterministic, no LLM needed |
| Translate one sentence | Single LLM call | No tools or loop |

---

## 17. Hands-On — Score Your Feature

Rate each signal 0–2 (0 = not present, 2 = strong). Total **6+** suggests multi-agent; **3–5** suggests hybrid; **0–2** stay single.

In [ ]:
def multi_agent_score(signals: dict[str, int]) -> str:
    total = sum(signals.values())
    if total >= 6:
        return f"strong multi-agent candidate (score {total})"
    if total >= 3:
        return f"consider hybrid or sequential multi (score {total})"
    return f"stay single-agent (score {total})"


FEATURES = {
    "Full onboarding guide": {
        "independent_subtasks": 2,
        "conflicting_policies": 2,
        "parallelism_value": 1,
        "governance_need": 2,
        "prompt_bloat": 1,
    },
    "Account setup FAQ": {
        "independent_subtasks": 0,
        "conflicting_policies": 0,
        "parallelism_value": 0,
        "governance_need": 0,
        "prompt_bloat": 0,
    },
    "Launch email with legal review": {
        "independent_subtasks": 2,
        "conflicting_policies": 2,
        "parallelism_value": 1,
        "governance_need": 2,
        "prompt_bloat": 2,
    },
}

for name, signals in FEATURES.items():
    print(f"{name}: {multi_agent_score(signals)}")

---

## 18. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Multi-agent by default | Higher cost, harder debugging | Start single; split on measured failures |
| Ignoring latency SLA | Poor UX in chat products | Single agent or parallel workers |
| No eval before split | "Architecture tourism" | Measure tool/routing errors first |
| Duplicate tools per agent | Inconsistent behavior | Shared tool layer |
| Critic without veto | Low-quality output still ships | Enforce reject → revise loop |
| Single agent with 20 tools | Tool selection chaos | Split by domain or reduce surface |

---

## 19. Optional — Live LLM Perspective

Set `USE_LIVE_LLM = True` with a valid API key for a one-sentence model answer on when multi-agent beats single-agent.

In [ ]:
USE_LIVE_LLM = False

OFFLINE_ANSWER = (
    "Multi-agent beats single-agent when roles need conflicting instructions, "
    "parallel specialists add value, or a dedicated critic must veto output before it ships."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "When does multi-agent beat single agent? One sentence."}],
            max_tokens=60,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_ANSWER)

---

## 20. Check Your Understanding

1. Name three dimensions where single-agent typically wins over multi-agent.
2. Why does multi-agent cost more in LLM calls even when the same tools are used?
3. What does the side-by-side comparison show for LLM calls in our onboarding demo?
4. When would you choose the **hybrid supervisor + subgraph** pattern?
5. What error rate in the migration simulator triggers extracting a compliance agent?
6. Why is *"Translate this sentence to French"* not a multi-agent candidate?
7. What is the recommended first step before splitting into multiple agents?

---

## 21. Summary

**Single-agent + tools** wins on cost, latency, and operational simplicity for linear tasks like doc Q&A and short tool chains. **Multi-agent** wins when roles conflict, work parallelizes, or a **critic/supervisor** must govern output before it ships.

**Key takeaways:**

- Use the **trade-off matrix** and **decision flowchart** before adding agents — not after.
- Prototype **single-agent** first; split when measurement shows tool selection failures, compliance misses, or unmaintainable prompts.
- **Hybrid supervisors + single-purpose subgraphs** are the common production end state.
- **Cost and latency** scale with LLM call count; multi-agent adds routing overhead but enables parallel specialists and governance.
- The **scoring rubric** (independent subtasks, conflicting policies, governance, etc.) gives a quick multi-agent candidacy check.

For any new feature, ask: *Can one focused agent with the right tools do this? If not, which role needs to be isolated — and is that worth the operational cost?*